# Entrenamiento en Colab (GPU T4, 16 GB)

Reproduce el entrenamiento principal del artículo con fp16 + acumulación de gradientes, que es lo que cabe en una T4.
Runtime → Change runtime type → **T4 GPU**. Duración estimada: varias horas (la T4 es bastante más lenta que la GPU del artículo; el tiempo en T4 real **no se midió**, ver README).

**Antes de ejecutar:** pon en `REPO` la URL del repositorio del equipo y deja los datos descargados según el README (sección Datos).

In [ ]:
REPO = 'REEMPLAZAR/url-del-repo'   # p. ej. https://github.com/usuario/traductorquechua.git
!git clone $REPO proyecto && cd proyecto && git checkout v1.0-final
%cd proyecto
!pip -q install -r requirements-colab.txt

In [ ]:
# Datos: limpieza y reparto sin fuga (descarga los parquet de Hugging Face a data/, ver README)
!python -X utf8 src/limpiar_datos.py && python -X utf8 src/marcar_religioso.py

In [ ]:
# Entrenamiento principal en T4: fp16, lote efectivo 32 (4 micro-lotes de 8), tope de 15 GB
!python -X utf8 src/finetune.py --nombre t4_s0 --semilla 0 --epocas 3 --batch 32 --lr 3e-5 \n    --precision fp16 --acumulacion 4 --limite-vram-gb 15

In [ ]:
# Si aparece OutOfMemory: sube --acumulacion a 8 y añade --checkpointing
import json; h = json.load(open('results/t4_s0/historial.json'))
print(h['gpu'], h['precision'], 'pico VRAM', h['pico_vram_gb'], 'GB', h['horas_gpu_entrenamiento'], 'h')
print(json.load(open('results/t4_s0/test.json')))